In [1]:
'''
seeing how many stars do not appear in the fits files
'''

'\nseeing how many stars do not appear in the fits files\n'

In [17]:
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
import precision_recall as pr
import gc  # garbage collector
import review_findings_functions as rff
import os
import importlib
import implantation_functions as imp
import corner
from astropy.wcs import WCS
import shutil
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy.signal import find_peaks
plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gist_gray"

In [ ]:
#okay so we need some files and some fits files
#well i guess we could do that at the end

In [ ]:
def implant_50_transients(
    image, sigmas, intensities, counter, norm
):
    """
    like it says on the tin

    """
    intensity_counter = counter
    transient_info = []  # empty list for adding info for implanted transients


    implanted = image    
    averages = np.average(image[35:-35,35:-35], axis=0) #cut off edges
    peaks, heights = find_peaks(averages, height=(None, None))
    min_height = np.sort(heights['peak_heights'])[-10]
    tall_map = heights['peak_heights']>=min_height
    tall_peaks = peaks[tall_map]+35
    
    

    for streak_x_val in tall_peaks:
        for iteration in range(0,5):
            
            center_value_y = np.random.randint(0,100)+35+iteration*(2900//5)
            intensity = intensities[intensity_counter]
            sigma = sigmas[intensity_counter]
            intensity_counter += 1
            implanted, maxcol = implant_transient_full_image(
                implanted, streak_x_val, center_value_y, intensity, sigma, norm
            )
            # print("intensity "+str(intensity))
            # print("xval "+str(streak_x_val))
            # if plot == True:
            #     show_streak(implanted, streak_x_val, center_value_y)
            trans_xcoord_ycoord_intensity_sigma = (
                maxcol, 
                center_value_y,
                intensity,
                sigma,
            )
            print("trans xcoord etc: ", trans_xcoord_ycoord_intensity_sigma)
            transient_info.append(trans_xcoord_ycoord_intensity_sigma)
    return implanted, transient_info

In [6]:
filename_example = 'ztf_20221020297847_000449_zr_c09_x_0013.'

In [32]:
def get_10_star_positions(image):
    averages = np.average(image[35:-35, :], axis=0) #cut off edges on y axis not x
    peaks, heights = find_peaks(averages, height=(None, None))
    min_height = np.sort(heights['peak_heights'])[-10]
    tall_map = heights['peak_heights']>=min_height
    tall_peaks = peaks[tall_map]
    return tall_peaks

In [64]:
def count_stars(filename):
    #i think we do all four quads at once
    xy_path = "/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/"
    data_path = '/lustre/lrspec/ZTF_data/2022/1020/'
    stars_in_manual_not_in_fits = 0
    stars_in_fits_not_in_manual = 0
    total_stars_in_manual = 0
    total_stars_in_fits = 0
    stars_in_both = 0
    filepath = data_path+filename+'fits.fz'
    with fits.open(filepath) as file:
        for quad in range(1,5):
            xy_file = xy_path+filename[:35]+str(quad)+".fits"
            # print(xy_file)
            image = file[quad].data   
            with fits.open(xy_file) as hdul_xy:
                xy_array = hdul_xy[1].data  
                # print(xy_array['X'])
                if len(xy_array)>25:
                    xy_array = xy_array[:25]
                total_stars_in_fits += len(xy_array)
                star_positions_manual = get_10_star_positions(image)
                # print(star_positions_manual)
                total_stars_in_manual += len(star_positions_manual)
                for position in star_positions_manual:
                    closest_idx = np.abs(xy_array['X'] - position).argmin()
                    # print(xy_array['X'][closest_idx], position)
                    if np.abs(xy_array['X'][closest_idx] - position) < 5:
                        stars_in_both+=1
                        xy_array['X'][closest_idx] = 6000 #nan was not working but it definitely wont match to 6000
                        # print("it worked")
                    else:
                        stars_in_manual_not_in_fits+=1
                        # print("it didnt work")
                        # print(position)
                stars_in_fits_not_in_manual += np.sum(xy_array['X']<6000) #should get rid of extras
                              
                # if np.sum(np.isclose(position, xy_array['X'], atol=5))>0: 
                    #a better version would be to match it to the closest and then drop that one if its <5
                
            
    
    return stars_in_manual_not_in_fits, total_stars_in_manual, stars_in_fits_not_in_manual, total_stars_in_fits, stars_in_both

In [ ]:
(5, 40, 17, 52, 35)

In [43]:
print(count_stars(filename_example))

/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020297847_000449_zr_c09_x_1.fits
/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020297847_000449_zr_c09_x_2.fits
/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020297847_000449_zr_c09_x_3.fits
it didnt work
200
it didnt work
772
it didnt work
1289
it didnt work
1413
/lustre/lrspec/ZTF_data/corrected_coordinates/xy_arrays/ztf_20221020297847_000449_zr_c09_x_4.fits
it didnt work
1470
(5, 40, 17, 52, 35)


In [ ]:
#if we had a list of files or a dataframe this would be eeeeasy
#where's my list for training random forest or something
#filenames_lcs.txt

In [49]:
'lightcurve_ztf_20221020138241_000686_zi_c01_x_0003_1_1783_1737.csv'[11:-16]

'ztf_20221020138241_000686_zi_c01_x_0003'

In [53]:
filenames_unique = []
lcfilenames = pd.read_csv('/home/3155/rwe/filenames_lcs.txt', sep=" ", header=None)
lcfilenames.columns = ["filename"]
for filename in lcfilenames['filename']:
    filename=filename[11:-16]+'.'
    filenames_unique.append(filename)
filenames_unique = np.unique(filenames_unique)

In [69]:
filenames_unique[::9]

array(['ztf_20221020138241_000686_zi_c01_x_0000.',
       'ztf_20221020138241_000686_zi_c10_x_0014.',
       'ztf_20221020141806_000686_zi_c02_x_0003.',
       'ztf_20221020141806_000686_zi_c02_x_0020.',
       'ztf_20221020141806_000686_zi_c11_x_0008.',
       'ztf_20221020141806_000686_zi_c11_x_0021.',
       'ztf_20221020148032_000686_zg_c03_x_0004.',
       'ztf_20221020148032_000686_zg_c03_x_0020.',
       'ztf_20221020148032_000686_zg_c12_x_0006.',
       'ztf_20221020148032_000686_zg_c12_x_0021.',
       'ztf_20221020154120_000640_zi_c04_x_0008.',
       'ztf_20221020154120_000640_zi_c04_x_0024.',
       'ztf_20221020154120_000640_zi_c13_x_0010.',
       'ztf_20221020154120_000640_zi_c13_x_0023.',
       'ztf_20221020157697_000640_zi_c05_x_0007.',
       'ztf_20221020157697_000640_zi_c05_x_0024.',
       'ztf_20221020157697_000640_zi_c14_x_0011.',
       'ztf_20221020157697_000640_zi_c14_x_0025.',
       'ztf_20221020163819_000640_zg_c06_x_0010.',
       'ztf_20221020163819_0006

In [76]:
df = pd.DataFrame([])

lcfilenames = pd.read_csv('/home/3155/rwe/filenames_lcs.txt', sep=" ", header=None)
lcfilenames.columns = ["filename"]
for i, filename in enumerate(filenames_unique[::9]):
    # print(filename)
    df.loc[i, 'filename']=filename
    # stars_in_manual_not_in_fits, total_stars_in_manual, stars_in_fits_not_in_manual, total_stars_in_fits, stars_in_both = count_stars(filename)
    cols = ['stars_in_manual_not_in_fits', 'total_stars_in_manual', 'stars_in_fits_not_in_manual', 'total_stars_in_fits', 'stars_in_both']
    df.loc[i, cols] = count_stars(filename)
    # break

In [79]:
np.sum(df['stars_in_manual_not_in_fits'])/np.sum(df['total_stars_in_manual'])

0.02857142857142857